# 6.2 TensorFlow / Keras to ONNX — Apply Notebook

## Objective

Convert TensorFlow and Keras models to ONNX using **tf2onnx**, validate correctness,
and benchmark inference against ONNX Runtime.

| # | Exercise | Key Skill |
|---|----------|-----------|
| 1 | Convert a Keras Sequential model | `tf2onnx.convert.from_keras` |
| 2 | Convert a CNN (Conv2D → MaxPool → Flatten → Dense) | Channels-last → ONNX |
| 3 | Numerical parity: TF vs ORT | Tolerance analysis |
| 4 | Dynamic batch handling | `input_signature` with `None` batch |
| 5 | SavedModel → ONNX conversion path | On-disk export |
| 6 | Inspect the converted graph | Ops, transposes, layout |
| 7 | Performance comparison | TF vs ORT latency |
| 8 | **Challenge:** convert a multi-branch Keras model | Functional API + parity |

> **Note:** All cells are guarded with `if HAS_TF:` so the notebook prints a
> helpful message instead of crashing when TensorFlow is not installed.

```
pip install tensorflow tf2onnx onnx onnxruntime numpy
```

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────
import os, sys, time, tempfile, warnings
warnings.filterwarnings("ignore")

import numpy as np
import onnx
from onnx import checker, TensorProto
import onnxruntime as ort

try:
    import tensorflow as tf
    import tf2onnx
    HAS_TF = True
    tf.get_logger().setLevel("ERROR")
    print(f"TensorFlow  : {tf.__version__}")
    print(f"tf2onnx     : {tf2onnx.__version__}")
except ImportError:
    HAS_TF = False
    print("TensorFlow or tf2onnx not installed.")
    print("Install with:  pip install tensorflow tf2onnx")
    print("Cells below will print a reminder and skip execution.")

print(f"ONNX        : {onnx.__version__}")
print(f"ORT         : {ort.__version__}")
print(f"NumPy       : {np.__version__}")

_SKIP_MSG = "⚠ TensorFlow not available — skipping this cell. Install with: pip install tensorflow tf2onnx"

## Exercise 1 — Convert a Keras Sequential Model

The simplest conversion path: build a `Sequential` model with `Dense` layers
and call `tf2onnx.convert.from_keras`.

A fully-connected network computes layer-by-layer:

$$\mathbf{h}^{(l)} = \sigma\!\left(W^{(l)} \mathbf{h}^{(l-1)} + \mathbf{b}^{(l)}\right)$$

where $\sigma$ is the activation function (ReLU here).

`tf2onnx.convert.from_keras` requires an **input signature** to know the
tensor shapes.  Setting `batch_size=None` allows dynamic batching.

In [ ]:
if HAS_TF:
    tf.keras.utils.set_random_seed(42)

    seq_model = tf.keras.Sequential([
        tf.keras.layers.InputLayer(input_shape=(16,)),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dense(32, activation="relu"),
        tf.keras.layers.Dense(10, name="logits"),
    ], name="simple_dense")

    seq_model.summary()

    with tempfile.TemporaryDirectory() as tmp:
        onnx_path = os.path.join(tmp, "simple_dense.onnx")
        spec = (tf.TensorSpec((None, 16), tf.float32, name="input"),)

        model_proto, _ = tf2onnx.convert.from_keras(seq_model, input_signature=spec, opset=17)
        onnx.save(model_proto, onnx_path)
        checker.check_model(model_proto)

        print(f"\nExport OK   — nodes: {len(model_proto.graph.node)}")
        print(f"File size   : {os.path.getsize(onnx_path)/1024:.1f} KB")
        print(f"Ops         : {sorted(set(n.op_type for n in model_proto.graph.node))}")

        # Quick parity
        sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
        x = np.random.randn(8, 16).astype(np.float32)
        tf_out = seq_model(x, training=False).numpy()
        ort_out = sess.run(None, {sess.get_inputs()[0].name: x})[0]
        diff = np.abs(tf_out - ort_out).max()
        assert diff < 1e-5, f"Parity FAILED: {diff}"
        print(f"Parity      : max|Δ| = {diff:.2e}  ✓")
else:
    print(_SKIP_MSG)

## Exercise 2 — Convert a CNN (Conv2D → MaxPool → Flatten → Dense)

TensorFlow uses **NHWC** (channels-last) layout by default, whereas ONNX
operators use **NCHW** (channels-first).  `tf2onnx` inserts `Transpose`
nodes to bridge the mismatch.

$$[B, H, W, C]_{\text{TF}} \xrightarrow{\text{Transpose}(0,3,1,2)} [B, C, H, W]_{\text{ONNX}}$$

The converter often folds redundant transposes, so the final graph is
more efficient than a naïve per-op conversion.

In [ ]:
if HAS_TF:
    tf.keras.utils.set_random_seed(0)

    cnn_model = tf.keras.Sequential([
        tf.keras.layers.InputLayer(input_shape=(28, 28, 1)),
        tf.keras.layers.Conv2D(16, 3, padding="same", activation="relu"),
        tf.keras.layers.MaxPooling2D(2),
        tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
        tf.keras.layers.MaxPooling2D(2),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dense(10, name="logits"),
    ], name="small_cnn_tf")

    cnn_model.summary()

    with tempfile.TemporaryDirectory() as tmp:
        onnx_path = os.path.join(tmp, "small_cnn_tf.onnx")
        spec = (tf.TensorSpec((None, 28, 28, 1), tf.float32, name="image"),)
        proto, _ = tf2onnx.convert.from_keras(cnn_model, input_signature=spec, opset=17)
        onnx.save(proto, onnx_path)
        checker.check_model(proto)

        n_transpose = sum(1 for n in proto.graph.node if n.op_type == "Transpose")
        print(f"\nExport OK  — {len(proto.graph.node)} nodes ({n_transpose} Transpose nodes for layout conversion)")
        print(f"Ops        : {sorted(set(n.op_type for n in proto.graph.node))}")
        print(f"File size  : {os.path.getsize(onnx_path)/1024:.1f} KB")

        # Verify shape
        sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
        inp = sess.get_inputs()[0]
        print(f"\nORT input  : name={inp.name!r}, shape={inp.shape}, type={inp.type}")
        out = sess.get_outputs()[0]
        print(f"ORT output : name={out.name!r}, shape={out.shape}, type={out.type}")
else:
    print(_SKIP_MSG)

## Exercise 3 — Numerical Parity: TensorFlow vs ORT

We run the same systematic parity suite used in the PyTorch chapter.

$$|y_{\text{TF}} - y_{\text{ORT}}|_{\infty} \;\le\; \epsilon \quad \forall\; \text{test inputs}$$

TF-to-ONNX conversions can introduce slightly larger deltas than PyTorch exports
because `tf2onnx` may decompose composite ops differently.

In [ ]:
if HAS_TF:
    with tempfile.TemporaryDirectory() as tmp:
        onnx_path = os.path.join(tmp, "parity_cnn.onnx")
        spec = (tf.TensorSpec((None, 28, 28, 1), tf.float32, name="image"),)
        proto, _ = tf2onnx.convert.from_keras(cnn_model, input_signature=spec, opset=17)
        onnx.save(proto, onnx_path)

        sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
        in_name = sess.get_inputs()[0].name

        cases = [
            ("randn  bs=1",  np.random.randn(1,  28, 28, 1).astype(np.float32)),
            ("randn  bs=8",  np.random.randn(8,  28, 28, 1).astype(np.float32)),
            ("randn  bs=32", np.random.randn(32, 28, 28, 1).astype(np.float32)),
            ("zeros  bs=4",  np.zeros((4, 28, 28, 1), dtype=np.float32)),
            ("ones   bs=4",  np.ones((4, 28, 28, 1), dtype=np.float32)),
            ("large  bs=4",  np.random.randn(4, 28, 28, 1).astype(np.float32) * 100),
            ("small  bs=4",  np.random.randn(4, 28, 28, 1).astype(np.float32) * 1e-3),
        ]

        print(f"{'─'*60}")
        print(f"  Parity report: TF CNN vs ORT")
        print(f"{'─'*60}")
        print(f"  {'Case':<18} {'max|Δ|':>10} {'mean|Δ|':>10}  Status")

        all_ok = True
        for label, x_np in cases:
            tf_out = cnn_model(x_np, training=False).numpy()
            ort_out = sess.run(None, {in_name: x_np})[0]
            d = np.abs(tf_out - ort_out)
            ok = np.allclose(tf_out, ort_out, atol=1e-5, rtol=1e-5)
            all_ok &= ok
            print(f"  {label:<18} {d.max():10.2e} {d.mean():10.2e}  {'PASS' if ok else 'FAIL'}")

        assert all_ok, "Parity check FAILED"
        print(f"\n  Overall: ALL PASSED")
else:
    print(_SKIP_MSG)

## Exercise 4 — Dynamic Batch Handling

Setting `batch_size=None` in the `TensorSpec` creates a symbolic dimension.
We verify that the ONNX model accepts any batch size at runtime.

In [ ]:
if HAS_TF:
    with tempfile.TemporaryDirectory() as tmp:
        onnx_path = os.path.join(tmp, "dynamic_batch.onnx")
        spec = (tf.TensorSpec((None, 28, 28, 1), tf.float32, name="image"),)
        proto, _ = tf2onnx.convert.from_keras(cnn_model, input_signature=spec, opset=17)
        onnx.save(proto, onnx_path)

        # Show the symbolic dim in the protobuf
        for inp in proto.graph.input:
            dims = []
            for d in inp.type.tensor_type.shape.dim:
                if d.dim_param:
                    dims.append(f"{d.dim_param} (dynamic)")
                else:
                    dims.append(str(d.dim_value))
            print(f"Input {inp.name!r}: [{', '.join(dims)}]")

        sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
        in_name = sess.get_inputs()[0].name

        print("\nRuntime batch sweep:")
        for bs in [1, 2, 4, 8, 16, 32, 64, 128]:
            x = np.random.randn(bs, 28, 28, 1).astype(np.float32)
            out = sess.run(None, {in_name: x})[0]
            assert out.shape[0] == bs
            print(f"  batch={bs:>4}  →  output shape {out.shape}")

        print("\nDynamic batch verified ✓")
else:
    print(_SKIP_MSG)

## Exercise 5 — SavedModel → ONNX Conversion Path

An alternative to `from_keras` is to first save the model to the **SavedModel**
format on disk, then convert with `tf2onnx.convert.from_saved_model`.  This is
useful when:

- You receive a pre-trained SavedModel from a colleague.
- The in-memory Keras API hits edge cases in tf2onnx.
- CI/CD pipelines serialize models between stages.

In [ ]:
if HAS_TF:
    with tempfile.TemporaryDirectory() as tmp:
        sm_dir = os.path.join(tmp, "saved_model")
        cnn_model.save(sm_dir, save_format="tf")
        print(f"SavedModel written to: {sm_dir}")
        sm_size = sum(
            os.path.getsize(os.path.join(dp, f))
            for dp, _, fns in os.walk(sm_dir) for f in fns
        )
        print(f"SavedModel size: {sm_size/1024:.1f} KB")

        onnx_path = os.path.join(tmp, "from_savedmodel.onnx")
        proto, _ = tf2onnx.convert.from_saved_model(
            sm_dir,
            input_signature=[tf.TensorSpec((None, 28, 28, 1), tf.float32, name="image")],
            opset=17,
        )
        onnx.save(proto, onnx_path)
        checker.check_model(proto)

        onnx_size = os.path.getsize(onnx_path)
        print(f"ONNX size       : {onnx_size/1024:.1f} KB")
        print(f"ONNX / SavedModel ratio: {onnx_size / sm_size:.2f}")

        # Parity between from_keras and from_saved_model
        sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
        in_name = sess.get_inputs()[0].name
        x = np.random.randn(4, 28, 28, 1).astype(np.float32)
        tf_out = cnn_model(x, training=False).numpy()
        ort_out = sess.run(None, {in_name: x})[0]
        diff = np.abs(tf_out - ort_out).max()
        assert diff < 1e-5
        print(f"Parity (SavedModel path): max|Δ| = {diff:.2e}  ✓")
else:
    print(_SKIP_MSG)

## Exercise 6 — Inspect the Converted Graph

Understanding how TF ops map to ONNX ops is important for debugging.

| TF Op | ONNX Op(s) |
|-------|------------|
| `Conv2D` | `Conv` (+ Transpose for layout) |
| `MaxPool2D` | `MaxPool` |
| `Dense` | `MatMul` + `Add` (or `Gemm`) |
| `Relu` | `Relu` |
| `Flatten` | `Reshape` |

In [ ]:
if HAS_TF:
    from collections import Counter

    with tempfile.TemporaryDirectory() as tmp:
        onnx_path = os.path.join(tmp, "inspect_tf.onnx")
        spec = (tf.TensorSpec((None, 28, 28, 1), tf.float32, name="image"),)
        proto, _ = tf2onnx.convert.from_keras(cnn_model, input_signature=spec, opset=17)
        onnx.save(proto, onnx_path)

        g = proto.graph
        init_names = {i.name for i in g.initializer}

        print("User Inputs:")
        for inp in g.input:
            if inp.name in init_names:
                continue
            tt = inp.type.tensor_type
            dims = [d.dim_param or str(d.dim_value) for d in tt.shape.dim]
            print(f"  {inp.name}: [{', '.join(dims)}]")

        print(f"\nOutputs:")
        for o in g.output:
            tt = o.type.tensor_type
            dims = [d.dim_param or str(d.dim_value) for d in tt.shape.dim]
            print(f"  {o.name}: [{', '.join(dims)}]")

        total_elems = sum(int(np.prod(i.dims)) for i in g.initializer)
        print(f"\nInitializers: {len(g.initializer)} tensors, {total_elems:,} elements")

        counts = Counter(n.op_type for n in g.node)
        print(f"\nOp distribution ({len(g.node)} nodes):")
        for op, c in counts.most_common():
            print(f"  {op:<20} {c:>3}  {'█' * c}")

        print(f"\nNode list:")
        for i, node in enumerate(g.node):
            ins = ', '.join(node.input[:3])
            print(f"  [{i:2d}] {node.op_type:<16} ({ins}) → {list(node.output)}")
else:
    print(_SKIP_MSG)

## Exercise 7 — Performance Comparison: TF vs ORT

We measure wall-clock latency for both runtimes across multiple batch sizes.

$$\text{speedup} = \frac{\text{median}_{\text{TF}}}{\text{median}_{\text{ORT}}}$$

In [ ]:
if HAS_TF:
    def bench(fn, warmup=30, iters=100):
        for _ in range(warmup):
            fn()
        times = []
        for _ in range(iters):
            t0 = time.perf_counter()
            fn()
            times.append(time.perf_counter() - t0)
        a = np.array(times) * 1000
        return {"median": np.median(a), "p95": np.percentile(a, 95), "mean": a.mean()}

    with tempfile.TemporaryDirectory() as tmp:
        onnx_path = os.path.join(tmp, "bench_tf.onnx")
        spec = (tf.TensorSpec((None, 28, 28, 1), tf.float32, name="image"),)
        proto, _ = tf2onnx.convert.from_keras(cnn_model, input_signature=spec, opset=17)
        onnx.save(proto, onnx_path)

        so = ort.SessionOptions()
        so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
        sess = ort.InferenceSession(onnx_path, so, providers=["CPUExecutionProvider"])
        in_name = sess.get_inputs()[0].name

        print(f"{'Batch':>6} {'TF median':>12} {'ORT median':>12} {'Speedup':>10}")
        print("─" * 44)

        for bs in [1, 4, 16, 64]:
            x = np.random.randn(bs, 28, 28, 1).astype(np.float32)

            tf_stats  = bench(lambda: cnn_model(x, training=False))
            ort_stats = bench(lambda: sess.run(None, {in_name: x}))
            speedup = tf_stats["median"] / ort_stats["median"]

            print(f"{bs:>6} {tf_stats['median']:>9.3f} ms {ort_stats['median']:>9.3f} ms {speedup:>9.2f}x")

        print("\n(Speedup > 1 means ORT is faster)")
else:
    print(_SKIP_MSG)

## Exercise 8 — Challenge: Convert a Multi-Branch Keras Model

Build a model with the **Functional API** that has two branches:

$$\text{branch}_1: \text{Conv}(3{\times}3) \to \text{ReLU}$$
$$\text{branch}_2: \text{Conv}(5{\times}5) \to \text{ReLU}$$
$$\text{merged} = \text{Concatenate}([\text{branch}_1, \text{branch}_2])$$

This tests tf2onnx's ability to handle non-sequential topologies.

In [ ]:
if HAS_TF:
    tf.keras.utils.set_random_seed(99)

    inp = tf.keras.Input(shape=(32, 32, 3), name="image")
    branch_a = tf.keras.layers.Conv2D(16, 3, padding="same", activation="relu", name="conv3x3")(inp)
    branch_b = tf.keras.layers.Conv2D(16, 5, padding="same", activation="relu", name="conv5x5")(inp)
    merged = tf.keras.layers.Concatenate(name="merge")([branch_a, branch_b])  # [B,32,32,32]
    x = tf.keras.layers.GlobalAveragePooling2D()(merged)
    out = tf.keras.layers.Dense(10, name="logits")(x)

    multi_branch = tf.keras.Model(inputs=inp, outputs=out, name="multi_branch")
    multi_branch.summary()

    with tempfile.TemporaryDirectory() as tmp:
        onnx_path = os.path.join(tmp, "multi_branch.onnx")
        spec = (tf.TensorSpec((None, 32, 32, 3), tf.float32, name="image"),)
        proto, _ = tf2onnx.convert.from_keras(multi_branch, input_signature=spec, opset=17)
        onnx.save(proto, onnx_path)
        checker.check_model(proto)

        print(f"\nExport OK — {len(proto.graph.node)} nodes")
        print(f"Ops: {sorted(set(n.op_type for n in proto.graph.node))}")

        # Concat should appear in the graph
        has_concat = any(n.op_type == "Concat" for n in proto.graph.node)
        print(f"Concat present: {has_concat}")
        assert has_concat, "Expected Concat node for multi-branch merge"

        # Parity
        sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
        in_name = sess.get_inputs()[0].name

        for bs in [1, 4, 16]:
            x = np.random.randn(bs, 32, 32, 3).astype(np.float32)
            tf_out = multi_branch(x, training=False).numpy()
            ort_out = sess.run(None, {in_name: x})[0]
            diff = np.abs(tf_out - ort_out).max()
            assert diff < 1e-4, f"Parity failed at bs={bs}: {diff}"
            print(f"  batch={bs:>2}  max|Δ| = {diff:.2e}  ✓")

        print("\nMulti-branch model parity verified.")
else:
    print(_SKIP_MSG)

## Summary

| Skill | Key Takeaway |
|-------|--------------|
| Sequential conversion | `tf2onnx.convert.from_keras` with an `input_signature` |
| CNN conversion | tf2onnx inserts `Transpose` nodes for NHWC → NCHW layout |
| Parity | Test across multiple input regimes; tf2onnx may decompose ops differently |
| Dynamic batch | Set `batch_size=None` in `TensorSpec` for runtime flexibility |
| SavedModel path | `from_saved_model` is a fallback when `from_keras` hits edge cases |
| Graph inspection | Understand how TF ops map to ONNX ops |
| Performance | ORT often outperforms TF for inference on CPU |
| Multi-branch | Functional API models with `Concatenate` convert cleanly |